In [1]:
import pandas as pd
from nltk import sent_tokenize
from nltk import word_tokenize
from tqdm import tqdm
from Levenshtein import ratio
import os

In [5]:
df=pd.read_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_all.feather')

In [6]:
df.shape

(53614, 6)

#### Sliding window of 3

In [ ]:
candidate_article_index=[]
candidate_window_index=[]
candidate=[]
candidate_ratio=[]
for article_index, article in tqdm(df['text'].items(), total=len(df['text'])):
    tokens = [w for w in word_tokenize(str(article)) if w.strip()]  
    tokens = [t.lower() for t in tokens]
    windows = [
        ' '.join(tokens[i:i+3])   
        for i in range(len(tokens) - 2)
    ]
    for window in windows:
        ratio_value = ratio(window, 'ku klux klan')
        if (ratio_value >= 0.8) and (ratio_value < 1.0): #0.7 includes articles like "klux klan and" (0.72) or "klux klan in" (0.75)
            candidate_article_index.append(article_index)
            candidate_window_index.append(windows.index(window))
            candidate.append(window)
            candidate_ratio.append(ratio_value)

100%|██████████| 53614/53614 [00:24<00:00, 2159.63it/s]


In [68]:
pd.DataFrame({'article_index': candidate_article_index, 'window_index': candidate_window_index, 'window': candidate, 'ratio': candidate_ratio})

,article_index,window_index,window,ratio
0,0,11,ku klux kl,0.909091
1,0,12,klux kl an,0.818182
2,2,138,ku klix kln,0.869565
3,2,194,ku klux klnn,0.916667
4,3,33,the kuklux klan,0.814815
...,...,...,...,...
1547,125,42,ku klux xian,0.833333
1548,156,15,ku klux klanism,0.888889
1549,104,73,ku klqx klan,0.916667
1550,155,32,mob kuklux klan,0.814815


In [56]:
candidate_article_index = []
candidate_window_index = []     # window index (same as token index)
candidate = []
candidate_index = []            # index of FIRST sliding-window start per article

for article_index, article in tqdm(df['text'].items(), total=len(df)):
    tokens = [w for w in word_tokenize(str(article)) if w.strip()]  

    first_token_idx = None  # store token index of first match

    for i in range(len(tokens) - 2):     # i = index of first token in window
        window = ' '.join(tokens[i:i+3])

        if ratio(window.lower(), 'ku klux klan') >= 0.8:

            candidate_article_index.append(article_index)
            candidate_window_index.append(i)   # the window starts at token i
            candidate.append(window)

            if first_token_idx is None:
                first_token_idx = i

    candidate_index.append(first_token_idx)

100%|██████████| 53614/53614 [00:24<00:00, 2196.54it/s]


In [57]:
pd.DataFrame({'article_index': candidate_article_index, 'window_index': candidate_window_index, 'window': candidate, 'first_token_index': candidate_index})

ValueError: All arrays must be of the same length

In [46]:
df.iloc[84]['text']

'TfIECAIULQgTHBNMaB V Pop Says If Sopers Cant Get Law Violators in St Paul the ku klux klan LAN CANDIDATE IS ELECTED MAYOR Fort Smith Kan A letter of con gratulation bearing the signatures of all the county and city officials of Haskell county was forwarded to Mayorelect David L Ford alleged Ku Klux Klan candidate who was elected mayor of Fort Smith by one of the largest majorities ever given a candidate The vote was 1824 to 524 A bitter fight had been waged against Ford and a day before the election large page circulars attack ing Ford and the Klan were distrib uted all over Fort Smith THREEMILE KLAN PARADE IN ILLINOIS Benton 111The citizens of John ston City West Frankfort and Ben ton were thrown into a quiver of ex citement by the invasion of a mon ster parade of Knights of the In visible Empire It was a demonstra tion to partially show the strength of the organization in this section Those who saw the parade declare it was three miles long and that nearly a thousand cars were in li

In [52]:
for index, item in enumerate(candidate_article_index):
    if item == 84:
        print(index)

0
74
149


In [55]:
candidate[149]

'ku klux klan'

In [ ]:
def keyword_search(df: pd.DataFrame, substring: str):
    state = []
    city = []
    date = []
    lccn = []
    sent_list = []
    title = []
    context_list = []
    text = []

    for idx, val in df.iterrows():
        tokenized_sent = word_tokenize(val['text'].lower())
        matching_items = [item for item in tokenized_sent if substring in item]
        to_context = []
        for num, sent in enumerate(tokenized_sent):
            interim_to_context = []
            if substring in sent:
                interim_to_context.extend(tokenized_sent[max(0, num - 20): num + 20])
                to_context.append(interim_to_context)

        state.append(val['state'])
        city.append(val['city'])
        date.append(val['date'])
        lccn.append(val['lCCN'])
        title.append(val['title'])
        sent_list.append(matching_items)
        context_list.append(to_context)
        text.append(val['text'])

    newdf = pd.DataFrame({'state': state, 'city': city, 'date': date, 'lccn': lccn, 'sent': sent_list, 'title': title, 'context': context_list, 'text': text})
    return newdf

In [ ]:
[item for item in word_tokenize(df['text'].iloc[0].lower()) if 'klan' in item]

In [ ]:
df['text'].iloc[100]

In [ ]:
path='/Volumes/T7/chroniclingamerica/kkk/revival/'
append_df=[]
for i in tqdm(os.listdir(path)):
    if i.startswith('._'):
        continue
    elif i.endswith('.csv'):
        df=pd.read_csv(path+i)
        df['text']=df['text'].astype(str)
        newdf=keyword_search(df, 'klan')
        lendf=newdf[newdf['sent'].apply(lambda x: len(x) > 0)]
        append_df.append(lendf)
        newdf=keyword_search(df, 'ku')
        lendf=newdf[newdf['sent'].apply(lambda x: len(x) > 0)]
        append_df.append(lendf)
        newdf=keyword_search(df, 'klux')
        lendf=newdf[newdf['sent'].apply(lambda x: len(x) > 0)]
        append_df.append(lendf)

In [ ]:
kkkdf=pd.concat(append_df).reset_index(drop=True)
kkkdf=kkkdf.explode(['sent', 'context'], ignore_index=True)

In [ ]:
kkkdf

In [ ]:
kkkdf.to_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_context')